# AREMA Steel Structure Load Rating - Implementation Notes

Python implementations of railroad bridge load rating calculations
including flexural stress, axial stress, combined stress checks,
fatigue evaluation, and 3D beam modeling with CadQuery.

Design provisions and procedures reference:

- AREMA. *Manual for Railway Engineering*, Chapter 15 -- Steel Structures.
- Lorak, S. *AREMA Railroad Bridge Load Rating Seminar*, 2017.

Refer to the current AREMA Manual for authoritative design provisions.


In [ ]:
from civilpy.general import units

In [ ]:
# Eq. 1.3.2a in python code
def eff_cyclic_stress_range(alpha, gamma, s_r):
    """
    Accepts the alpha factor as well as a list of gamma and s_ri values to 
    calculate the effective cyclic stress range for the total number of cycles

    :param: alpha = (float) alpha value from AREMA Table 15-7-1
    :param: gamma = (list) a list of the gamma values, defined in AREMA 
        9.1.3.13
    :param: s_r = (list) a list of the s_r values, AREMA 9.1.3.13

    :returns: stress_range - the effective cyclic stress range
    """
    stress_range = 0
    for i, value in enumerate(gamma):
        stress_range += alpha * (gamma * s_ri ** 3) ** (1/3)
    return stress_range

In [ ]:
# Alpha factor Table
alpha_factor = {
    'hanger': 1.15,
    'other_truss_member': 0.87,
    'stringer_girder_fb_under_10_ft': 1.00,
    'string_girder_fb_over_10_ft': 0.85
}

In [ ]:
def speed_impact_reduction(speed):
    """
    Uses the speed of the line as an input, returns the reduction factor based
    on the speed of the line. 60 mph returns a factor of 1.0, 10 mph returns 
    0.2
    """
    factor = 1 - (0.8 / 2500) * (60 - speed) ** 2

    if factor > 0.2:
        return factor
    else:
        return 0.2

In [ ]:
x = speed_impact_reduction(10)

In [ ]:
speed_impact_reduction(40)

In [ ]:
speed_impact_reduction(60)

In [ ]:
# Dead Load
rail_weight = 200 * units('lbf/ft')
timber_weight = 600 * units('lbf/ft')
guard_timber = 26.67 * units('lbf/ft')

total_deck_weight = rail_weight + timber_weight + guard_timber
deck_weight_per_girder = round((
                                400*units('lbf/ft') + total_deck_weight / 2
                               ).to('kip/ft'), 2)

In [ ]:
deck_weight_per_girder

In [ ]:
# Bending Moment at Mid-Span due to Dead Load M = wL^2/8
dl_mom = round((deck_weight_per_girder * (49*units('ft')) ** 2) / 8, 0)
print(f"{dl_mom=:}")

In [ ]:
# Live Load (1/2 track AREMA Table 15-1-16)
live_load = 1841.68 * units('ft*kip')

In [ ]:
# Impact Load (For L < 80')
# AREMA 15.1.3.5.a: I = (40 - 3*L^2/1600) where L is span length in ft (result is %)
L_ft = 49  # span length in ft
vert_imp_effect = round(40 - 3 * L_ft**2 / 1600, 2)
rock_effect = round(5 / 9 * 20, 2)

In [ ]:
print(f"{vert_imp_effect=:}")
print(f"{rock_effect=:}")

In [ ]:
# Impact Reduction Factor
max_speed = 50
impact_reduction_factor = round(1 - (0.8/2500) * (60-max_speed)**2, 2)
if impact_reduction_factor > 0.2:
    pass
else:
    impact_reduction_factor = 0.2

print(f"{impact_reduction_factor=:}")

In [ ]:
total_impact = round(rock_effect + vert_imp_effect * 
                     impact_reduction_factor, 2)
print(f"{total_impact=:}")

In [ ]:
live_load_impact_moment = round(total_impact / 100 * live_load, 2)
print(f"{live_load_impact_moment=:}")

In [ ]:
total_moment = round(live_load + live_load_impact_moment, 0)
print(f"{total_moment=:}")

In [ ]:
# Define Section Properties - Mid-Span
girder_depth = 67.50
girder_na = 32.25
I_xx_gross = 58_066 * units('in^4')
I_xx_net = 48_349 * units('in^4')
S_xx_top_gross = 1_647 * units('in^3')
S_xx_bot_gross = 1_800 * units('in^3')
S_xx_top_net = 1_372 * units('in^3')
S_xx_bot_net = 1_499 * units('in^3')

In [ ]:
# Bending Stresses - Compressive
round((dl_mom * 12 * units.inch/units.foot) / S_xx_top_gross, 2)

In [ ]:
# Bending Stresses - Tensile
round((dl_mom * 12 * units.inch/units.foot) / S_xx_bot_net, 2)

In [ ]:
# LL + Impact - Compressive
round((total_moment * 12 * units.inch/units.foot) / S_xx_top_gross, 2)

In [ ]:
# LL + Impact - Tensile
round((total_moment * 12 * units.inch/units.foot) / S_xx_bot_net, 2)

In [ ]:
# Dead Load
dl = -33 * units.kips
ll_comp = -158 * units.kips
ll_ten = 75 * units.kips
length = 149.50 * units.ft

In [ ]:
# For L >= 80'
vert_imp_eff = round(16 + 600*units('ft')/(length-30*units('ft')), 2)
print(f"{vert_imp_eff=:}")

In [ ]:
rock_eff = round(5/21*20, 2)
print(f"{rock_eff=:}")

In [ ]:
# Rail speed = 60 mph meaning impact can't be reduced
impact_effect = vert_imp_eff + rock_eff
print(f"{impact_effect=:}")

In [ ]:
# Impact Compression Load
impact_c = round(ll_comp * impact_effect/100, 2)
print(f"{impact_c=:}")

In [ ]:
# Impact Tension Load
impact_t = round(ll_ten * impact_effect/100, 2)
print(f"{impact_t=:}")

<u>**Total Live Load + Impact Load**</u>

In [ ]:
# Compression
ll_imp_comp = round(ll_comp + impact_c, 0)
print(f"{ll_imp_comp=:}")

In [ ]:
# Tension
ll_imp_ten = round(ll_ten + impact_t, 0)
print(f"{ll_imp_ten=:}")

<u>**Section Properties**</u>

In [ ]:
gross_area = (19 + 12 + 16.25) * units('in^2')
print(f"{gross_area=:}")

In [ ]:
# Net Area
length_abc = round(9.5 - (1*1), 2)
length_abde = round(9.5 - 2*1 + ((1.75**2)/(4*5.50)), 2)
controlling_length = min(length_abc, length_abde)
print(f"{controlling_length}")

In [ ]:
# //TODO - Dig into these numbers more
# 4L5x5x1/2
net_5x5x_5 = 4*7.64*.5

if ((net_5x5x_5 / 2 * 0.5) / 4.75) * 100 < 85:
    print('OK')
else:
    print('NG')

In [ ]:
# 2 PL 1/2 x 22 w/ 10" PERF
net_1_2x22 = 2 * (0.5 * (22-10) - .5*(2*1))

if net_1_2x22 / 12 * 100 < 85:
    print('OK')
else:
    print('NG')

In [ ]:
# 2 PL 5/8 x 13
net_5_8x13 = 2 * (0.625 * 13 - .625*(4*1))

if net_5_8x13 / 16.25 * 100 < 85:
    print('OK')
else:
    print('NG')

In [ ]:
A_n = (net_5_8x13 + net_1_2x22 + net_5x5x_5) * units('in^2')
print(f"{A_n}")

<u>Axial Stresses:</u>

In [ ]:
# Dead Load - Axial Compressive Stress
round(dl / gross_area, 2)

In [ ]:
# Live Load + Impact - Axial Compressive Stress
round(ll_imp_comp / gross_area, 2)

In [ ]:
# Live Load + Impact - Axial Compressive Stress
round(ll_imp_ten / A_n, 2)

In [ ]:
# Dead Load
axial_compressive_stress = round((-172 * units.kips) / (58.50 * units('in^2')), 2)
print(f"{axial_compressive_stress =:}")

# LL + Impact
ll_imp_axial_stress = round((-516 * units.kips) / (58.50 * units('in^2')), 2)
print(f"{ll_imp_axial_stress =:}")

# Wind Load
wind_axial_stress = round((-28 * units.kips) / (58.50 * units('in^2')), 2)
print(f"{wind_axial_stress =:}")

In [ ]:
# Wind Load
wind_bending_stress = round((-136 * units('ft*kips')).to('in*kips') / 
                          (562.21 * units('in^3')), 2)
print(f"{wind_bending_stress =:}")

In [ ]:
# Combined Stress
combined_stress = axial_compressive_stress + ll_imp_axial_stress + \
                    wind_axial_stress + wind_bending_stress

print(f"{combined_stress =:}")

# Consolidated Python Approach

## Inputs/Summary

In [ ]:
from civilpy.structural.steel import W, L

In [ ]:
import cadquery as cq

In [ ]:
import itertools

In [ ]:
# Define the bottom flange plate
beam_length = 12.*11.
flange_width = 16.0
flange_thickness = 0.375

# Define the Web Variables
web_height = 33.5
web_thickness = 0.6550

flange_angle_height = 6
flange_angle_width = 6
flange_angle_thickness = 5/8

# Start removing Rivet Holes
rivet_hole_dia = 7/8
flange_rivets_no_rows = 2
flange_rivets_gage = 2
flange_rivets_pitch = 2

web_rivets_rows = 2
web_rivets_gage = 2
web_rivets_pitch = 2

## Object Creation

In [ ]:
# create the top flange
top_flange = (
    cq.Workplane("XY")
    .box(flange_width, beam_length, flange_thickness)
).translate((0,0,flange_thickness+web_height))

# Add the web
web = (
    cq.Workplane("XY")
    .box(
    web_thickness, beam_length, web_height
).translate((
    0,
    0,
    web_height/2+flange_thickness/2
)))

# Add the bot flange
bot_flange = cq.Workplane("XY").box( 
    flange_width, beam_length, flange_thickness
).translate((0, 0, 0))

beam = bot_flange.newObject(bot_flange.objects + web.objects + top_flange.objects)

In [ ]:
# Add the first set of flange angles
flange_angle = (
    cq.Workplane("XZ")
    .box(flange_angle_width, flange_angle_thickness, beam_length)
    .translate(
        (web_thickness/2+flange_angle_width/2, 
         0, 
         flange_thickness*0.5+flange_angle_thickness/2)
    )
    .union(
        cq.Workplane("XZ")
        .box(flange_angle_thickness, flange_angle_height, beam_length)
        .translate(
            (web_thickness/2+flange_angle_thickness/2, 
             0, 
             flange_thickness*0.5+flange_angle_height/2)
        )
    )
)

In [ ]:
# Mirror the flange angles twice to put one at each corner of the web
flange_angle2 =  flange_angle.mirror("YZ")
bottom_angles = flange_angle.newObject([flange_angle.objects[0], flange_angle2.objects[0]])
top_angles = bottom_angles.mirror("XY").translate((0,0, flange_thickness + web_height))

all_angles = beam.newObject(bottom_angles.objects+top_angles.objects)

flat_list = all_angles.objects + beam.objects

# Add the flange angles to the beam
beam = all_angles.newObject(flat_list).translate((0,beam_length/2,0))

In [ ]:
beam

In [ ]:
# Find the center of the beam
beam_center_y = beam_length / 2
beam_center_x = flange_width / 2

# Create a workplane on top of the box and start adding holes
beam_with_holes = beam.faces(">Z").workplane()

# Calculate the number of holes to fit into the length minus the offset on both sides
num_holes = int((beam_length - rivet_hole_dia) // flange_rivets_pitch)

# Add holes along the length of the girder - +X axis
for i in range(num_holes - 1):
    y_position = (i * flange_rivets_pitch) + flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(flange_width/4 - flange_rivets_gage/2, y_position)]).hole(rivet_hole_dia)

for i in range(num_holes):
    y_position = (i * flange_rivets_pitch) + 0.5 * flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(flange_width/4 + flange_rivets_gage/2, y_position)]).hole(rivet_hole_dia)

# Add holes along the length of the girder - -X axis
for i in range(num_holes - 1):
    y_position = (i * flange_rivets_pitch) + flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(-flange_width/4 + flange_rivets_gage/2, y_position)]).hole(rivet_hole_dia)

for i in range(num_holes):
    y_position = (i * flange_rivets_pitch) + 0.5 * flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(-flange_width/4 - flange_rivets_gage/2, y_position)]).hole(rivet_hole_dia)

In [ ]:
# Web holes - Top
beam_with_holes = beam_with_holes.faces(">X").workplane()

# Bottom Flange angle to web holes
for i in range(num_holes):
    y_position = (i * flange_rivets_pitch) + 0.5 * flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(y_position, -web_height + flange_rivets_gage/2 + flange_rivets_pitch*.5)]).hole(rivet_hole_dia)

for i in range(num_holes-1):
    y_position = (i * flange_rivets_pitch) + flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(y_position, -web_height + flange_rivets_gage/2 + flange_rivets_pitch*1.5)]).hole(rivet_hole_dia)

In [ ]:
# Top Flange angle to web holes
for i in range(num_holes):
    y_position = (i * flange_rivets_pitch) + 0.5 * flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(y_position, - flange_rivets_pitch*1.5)]).hole(rivet_hole_dia)

for i in range(num_holes - 1):
    y_position = (i * flange_rivets_pitch) + flange_rivets_pitch + flange_rivets_pitch / 2
    beam_with_holes = beam_with_holes.pushPoints([(y_position, -2 * flange_rivets_gage - flange_rivets_pitch/2)]).hole(rivet_hole_dia)

In [ ]:
beam_with_holes

# Exporting to Various Formats

When converting to .step/.stp format, it's 

In [ ]:
beam_with_holes = beam_with_holes.objects[0].scale(25.4)

In [ ]:
cq.exporters.export(beam_with_holes, 'C:\\Users\\dane.parks\\3D Objects\\beam.stp')

In [ ]:
beam_with_holes

# DXF

.dxf files are 2D files, meaning that a workplane has to be specified in order to understand which view is being exported to the file.

In [ ]:
face = beam_with_holes.faces(">Z")

In [ ]:
print(face)

In [ ]:
beam_with_holes

In [ ]:
cross_section = cq.Workplane(cube, "XZ").box(10, 10).intersect(cube) 

In [ ]:
import os
from pathlib import Path
# from civilpy.general.jupyter import notebook_to_pdf

In [ ]:
import asyncio
import nbformat
from nbconvert import WebPDFExporter
from nbconvert.preprocessors import TagRemovePreprocessor

In [ ]:
def notebook_to_pdf(notebook_path):
    # Set the appropriate event loop policy for Windows
    if asyncio.get_event_loop().is_running():
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

    # Read the notebook
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = nbformat.read(f, as_version=4)

    # Configure the tag removal preprocessor
    tag_remove_preprocessor = TagRemovePreprocessor()
    tag_remove_preprocessor.remove_cell_tags = ("remove_cell",)
    tag_remove_preprocessor.remove_single_output_tags = ("remove_output",)
    tag_remove_preprocessor.remove_input_tags = ("remove_input",)

    # Create the WebPDF exporter and register the preprocessor
    pdf_exporter = WebPDFExporter()
    pdf_exporter.register_preprocessor(tag_remove_preprocessor, enabled=True)

    # Convert the notebook to PDF
    pdf_data, resources = pdf_exporter.from_notebook_node(notebook)

    # Save the PDF to a file
    pdf_filename = notebook_path.replace(".ipynb", ".pdf")
    with open(pdf_filename, "wb") as f:
        f.write(pdf_data)
    print(f"PDF created: {pdf_filename}")

In [ ]:
path = str(Path(os.getcwd()) / 'AREMA Steel Structure Load Rating.ipynb')

In [ ]:
notebook_to_pdf(path)